In [ ]:
import cv2
import numpy as np
from PIL import Image
import json

class CoordinateCalibrator:
    def __init__(self):
        self.coordinates = {}
        self.current_region = None
        self.start_point = None
        self.end_point = None
        self.drawing = False
        
    def mouse_callback(self, event, x, y, flags, param):
        """Mouse callback for selecting regions"""
        if event == cv2.EVENT_LBUTTONDOWN:
            self.drawing = True
            self.start_point = (x, y)
            
        elif event == cv2.EVENT_MOUSEMOVE:
            if self.drawing:
                self.end_point = (x, y)
                
        elif event == cv2.EVENT_LBUTTONUP:
            self.drawing = False
            self.end_point = (x, y)
            if self.current_region:
                self.coordinates[self.current_region] = {
                    'x1': min(self.start_point[0], self.end_point[0]),
                    'y1': min(self.start_point[1], self.end_point[1]),
                    'x2': max(self.start_point[0], self.end_point[0]),
                    'y2': max(self.start_point[1], self.end_point[1])
                }
                print(f"Saved {self.current_region}: {self.coordinates[self.current_region]}")
    
    def calibrate_regions(self, image_path):
        """Interactive calibration tool"""
        # Load image
        img = cv2.imread(image_path)
        original_img = img.copy()
        
        regions_to_calibrate = [
            'date_area',
            'day_area', 
            'showroom_area',
            'main_table_area',
            'names_column',
            'sales_order_column',
            'amount_column',
            'reason_column'
        ]
        
        print("Coordinate Calibration Tool")
        print("Instructions:")
        print("- Click and drag to select each region")
        print("- Press 'n' for next region")
        print("- Press 's' to save coordinates")
        print("- Press 'q' to quit")
        print("-" * 50)
        
        region_index = 0
        
        while region_index < len(regions_to_calibrate):
            self.current_region = regions_to_calibrate[region_index]
            print(f"\nSelect region: {self.current_region}")
            
            # Create a copy for drawing
            display_img = original_img.copy()
            
            # Draw existing regions
            for region_name, coords in self.coordinates.items():
                cv2.rectangle(display_img, 
                            (coords['x1'], coords['y1']), 
                            (coords['x2'], coords['y2']), 
                            (0, 255, 0), 2)
                cv2.putText(display_img, region_name, 
                           (coords['x1'], coords['y1']-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            
            cv2.namedWindow('Calibration', cv2.WINDOW_NORMAL)
            cv2.setMouseCallback('Calibration', self.mouse_callback)
            
            while True:
                temp_img = display_img.copy()
                
                # Draw current selection
                if self.drawing and self.start_point and self.end_point:
                    cv2.rectangle(temp_img, self.start_point, self.end_point, (255, 0, 0), 2)
                
                cv2.imshow('Calibration', temp_img)
                key = cv2.waitKey(1) & 0xFF
                
                if key == ord('n'):  # Next region
                    region_index += 1
                    break
                elif key == ord('b'):  # Back to previous region
                    region_index = max(0, region_index - 1)
                    break
                elif key == ord('s'):  # Save and continue
                    self.save_coordinates('coordinates.json')
                    print("Coordinates saved!")
                elif key == ord('q'):  # Quit
                    cv2.destroyAllWindows()
                    return self.coordinates
        
        cv2.destroyAllWindows()
        self.save_coordinates('coordinates.json')
        return self.coordinates
    
    def save_coordinates(self, filename):
        """Save coordinates to JSON file"""
        with open(filename, 'w') as f:
            json.dump(self.coordinates, f, indent=2)
        print(f"Coordinates saved to {filename}")
    
    def load_coordinates(self, filename):
        """Load coordinates from JSON file"""
        try:
            with open(filename, 'r') as f:
                self.coordinates = json.load(f)
            print(f"Coordinates loaded from {filename}")
            return self.coordinates
        except FileNotFoundError:
            print(f"File {filename} not found")
            return {}

# Usage
if __name__ == "__main__":
    calibrator = CoordinateCalibrator()
    
    # Calibrate using sample image
    sample_image = "nova_data/printed/2.jpg"
    coordinates = calibrator.calibrate_regions(sample_image)
    
    print("\nFinal coordinates:")
    for region, coords in coordinates.items():
        print(f"{region}: {coords}")

Coordinate Calibration Tool
Instructions:
- Click and drag to select each region
- Press 'n' for next region
- Press 's' to save coordinates
- Press 'q' to quit
--------------------------------------------------

Select region: date_area
Saved date_area: {'x1': 281, 'y1': 313, 'x2': 517, 'y2': 358}
Coordinates saved to coordinates.json
Coordinates saved!

Select region: day_area
Saved day_area: {'x1': 517, 'y1': 308, 'x2': 653, 'y2': 355}
Coordinates saved to coordinates.json
Coordinates saved!

Select region: showroom_area
Saved showroom_area: {'x1': 1108, 'y1': 220, 'x2': 1963, 'y2': 301}
Coordinates saved to coordinates.json
Coordinates saved!

Select region: main_table_area
Saved main_table_area: {'x1': 195, 'y1': 336, 'x2': 2974, 'y2': 2130}
Coordinates saved to coordinates.json
Coordinates saved!

Select region: names_column
Saved names_column: {'x1': 297, 'y1': 358, 'x2': 487, 'y2': 1332}
Coordinates saved to coordinates.json
Coordinates saved!

Select region: sales_order_column